# ReconGuard — ad-hoc exploration

Not part of the pipeline. Used to eyeball the generated source data and the
shape of the breaks the reconciliation engine is expected to catch, before
writing assertions in `tests/`. Uses only the standard library so it runs
with nothing beyond `requirements.txt`.

Run `make generate` first so `data/raw/hris.csv` and `data/raw/payroll.csv` exist.

In [ ]:
import csv
from collections import Counter

def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

hris = read_csv("../data/raw/hris.csv")
payroll = read_csv("../data/raw/payroll.csv")

print(f"hris rows: {len(hris)}   payroll rows: {len(payroll)}")
hris[:3]

In [ ]:
payroll[:3]

## Sanity-check the injected breaks

`data_generator/break_injector.py` drops payroll rows, adds orphan payroll
rows, perturbs `gross_pay`, and duplicates HRIS employees with a stale
effective-dated record. Confirm those show up before trusting the
reconciliation engine's counts.

In [ ]:
hris_ids = {r["employee_id"] for r in hris}
payroll_ids = {r["employee_id"] for r in payroll}

orphan_payroll = payroll_ids - hris_ids       # missing_left candidates
missing_payroll = hris_ids - payroll_ids      # missing_right candidates
id_counts = Counter(r["employee_id"] for r in hris)
duplicated = {emp_id for emp_id, n in id_counts.items() if n > 1}

print(f"orphan payroll employee_ids (in payroll, not hris): {sorted(orphan_payroll)}")
print(f"employees with no payroll row (in hris, not payroll): {sorted(missing_payroll)}")
print(f"duplicate hris employee_ids (stale effective-dated rows): {sorted(duplicated)}")

In [ ]:
# gross_pay vs. expected (base_salary / 12) to eyeball the perturbed rows
current_salary = {r["employee_id"]: float(r["base_salary"])
                  for r in hris if r["is_current"] == "True"}

mismatches = []
for row in payroll:
    base = current_salary.get(row["employee_id"])
    if base is None:
        continue
    expected = round(base / 12, 2)
    delta = round(float(row["gross_pay"]) - expected, 2)
    if abs(delta) > 1.0:
        mismatches.append((row["employee_id"], row["gross_pay"], expected, delta))

mismatches.sort(key=lambda r: -abs(r[3]))
mismatches

## Department headcount, for a feel of the base data

(Mirrors `pipelines/gold/headcount_by_dept.py`, computed here with plain
Python instead of Spark since this notebook is a quick-look tool, not a
pipeline stage.)

In [ ]:
by_dept = {}
for r in hris:
    if r["is_current"] != "True":
        continue
    dept = r["department"]
    count, total = by_dept.get(dept, (0, 0.0))
    by_dept[dept] = (count + 1, total + float(r["base_salary"]))

for dept, (count, total) in sorted(by_dept.items()):
    print(f"{dept:<12} headcount={count:<4} avg_base_salary={round(total / count, 2)}")